[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/06_calibration_uncertainty/06_calibration_uncertainty.ipynb)

# 06 · 校准与不确定性（从零实现）

目标：把「模型的置信度说话算数吗」变成可测、可修的量。用 numpy 从零：① 造一个**过度自信**的分类器；② **ECE/MCE** 分箱从零；③ **可靠性图**数据；④ **温度缩放**（拟合 T、ECE 下降、argmax 不变）；⑤ **Brier 分数 + Murphy 三分解**（精确验证 Brier=rel−res+unc）；⑥ **选择性预测** risk-coverage。

路线：过度自信预测 → ECE/MCE → 可靠性图 → 温度缩放 → Brier 三分解 → risk-coverage → ✏️ 练习 → 📖 答案 → 🧪 真实数据校准胶囊。

> 纪律：所有随机用 `default_rng(seed)`；所有「应成立的性质」写 `assert`（过度自信集 ECE>0、温度缩放后 ECE↓ 且 accuracy 不变、完美校准 ECE≈0、Brier 三项相加=分箱 Brier）。

## 0 · 数据 helper（联网取真实数据，失败回退）

下面这个 cell 定义全课统一的下载工具，最后的真实数据胶囊会用到。

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 1 · 造一个过度自信的分类器

校准研究的对象是「预测概率 vs 真实正确频率」。先造一个**过度自信**的多分类预测：
1. 用一组**中等区分度**的真 logits 生成真标签（标签来自真概率）；
2. 模型上报的 logits 是把真 logits **锐化**（乘一个 >1 的因子）的结果 —— 这正是交叉熵过拟合导致的「概率过尖」。

锐化因子 `sharpen` 就是「该用多大温度去修」的真值，后面温度缩放应当把它恢复出来。

In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def make_overconfident(n=6000, K=4, sharpen=2.5, seed=0):
    '''返回 (logits_model, y, probs_true)。
       真标签来自真概率 softmax(true_logits)；模型 logits = true_logits*sharpen (过度自信)。'''
    r = np.random.default_rng(seed)
    true_logits = r.normal(0, 1.0, (n, K))      # 中等区分度
    probs_true = softmax(true_logits)
    y = np.array([r.choice(K, p=probs_true[i]) for i in range(n)])
    logits_model = true_logits * sharpen        # 锐化 -> 过度自信
    return logits_model, y, probs_true

logits, y, _ = make_overconfident(sharpen=2.5, seed=0)
probs = softmax(logits)
conf = probs.max(1); pred = probs.argmax(1); correct = (pred == y).astype(float)
acc = correct.mean(); avg_conf = conf.mean()
print(f'accuracy      = {acc:.3f}')
print(f'平均置信度    = {avg_conf:.3f}')
print(f'置信度 - 准确率 = {avg_conf - acc:+.3f}  (>0 即过度自信)')
assert avg_conf > acc + 0.1, '构造的模型应明显过度自信(平均置信度 >> 准确率)'
print('✅ 造出一个过度自信的分类器：嘴上 0.9，实际只对 0.x —— 这就是要修的病')

## 2 · ECE 与 MCE：从零分箱

把预测按置信度等宽分箱，每箱算 |准确率 − 平均置信度|：
$$\mathrm{ECE}=\sum_b \frac{n_b}{N}\,|\mathrm{acc}(b)-\mathrm{conf}(b)|,\qquad \mathrm{MCE}=\max_b |\mathrm{acc}(b)-\mathrm{conf}(b)|$$

过度自信的模型 ECE 应明显 > 0。

In [ ]:
def calibration_bins(conf, correct, n_bins=15):
    '''等宽分箱，返回每个非空箱的 (平均置信度, 准确率, 样本数)。'''
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        out.append((conf[m].mean(), correct[m].mean(), int(m.sum())))
    return out

def ece(conf, correct, n_bins=15):
    N = len(conf); e = 0.0
    for conf_b, acc_b, n_b in calibration_bins(conf, correct, n_bins):
        e += n_b / N * abs(acc_b - conf_b)
    return e

def mce(conf, correct, n_bins=15):
    gaps = [abs(acc_b - conf_b) for conf_b, acc_b, _ in calibration_bins(conf, correct, n_bins)]
    return max(gaps) if gaps else 0.0

e = ece(conf, correct); m = mce(conf, correct)
print(f'ECE = {e:.4f}   MCE = {m:.4f}')
# 对拍: ECE 也可由「按样本的 |该样本所在箱的acc - 该样本conf|」加权得到 —— 这里直接核对量级
assert e > 0.05, '过度自信模型 ECE 应明显 > 0'
assert m >= e, 'MCE(最坏箱) 不应小于 ECE(平均)'
print('✅ ECE 量出了校准缺口；MCE>=ECE（最坏箱 >= 平均），过度自信被抓个正着')

## 3 · 可靠性图：把校准画出来

可靠性图 = 每个箱画一个点 (x=平均置信度, y=准确率)，对角线 y=x 是完美校准。过度自信的模型，点会**整体落在对角线下方**（置信度 > 准确率）。用文本展示（matplotlib 可选）。

In [ ]:
bins_info = calibration_bins(conf, correct, n_bins=10)
print(f"{'箱':>3} {'平均置信度':>10} {'准确率':>8} {'差距':>8} {'样本数':>7}")
below = 0
for k, (conf_b, acc_b, n_b) in enumerate(bins_info):
    gap = conf_b - acc_b
    flag = '过度自信' if gap > 0.02 else ('不足自信' if gap < -0.02 else '~校准')
    print(f'{k:>3} {conf_b:>10.3f} {acc_b:>8.3f} {gap:>+8.3f} {n_b:>7}  {flag}')
    if gap > 0:
        below += 1
print(f'\n{below}/{len(bins_info)} 个箱落在对角线下方(置信度>准确率) = 过度自信')
assert below >= len(bins_info) * 0.6, '过度自信模型多数箱应在对角线下方'

# 可选: 画图（在 Jupyter 里会内联显示；无 GUI/headless 时自动跳过，绝不阻塞）
try:
    import matplotlib
    if 'inline' not in matplotlib.get_backend().lower():
        matplotlib.use('Agg')        # 非交互后端: plt.show() 不会阻塞
    import matplotlib.pyplot as plt
    xs = [b[0] for b in bins_info]; ys = [b[1] for b in bins_info]
    plt.figure(figsize=(4,4)); plt.plot([0,1],[0,1],'k--',label='perfect')
    plt.plot(xs, ys, 'o-', label='model'); plt.xlabel('confidence'); plt.ylabel('accuracy')
    plt.legend(); plt.title('reliability diagram'); plt.show(); plt.close('all')
except Exception as _e:
    print('(跳过画图:', type(_e).__name__, '—— 上面的文本表已说明问题)')
print('✅ 可靠性图: 曲线压在对角线下方，肉眼可见的过度自信')

## 4 · 温度缩放：一个标量修好校准

在**验证集**上拟合温度 T，最小化 NLL：`T* = argmin_T −Σ log softmax(z/T)[y]`。

关键性质要全部验证：(a) 过度自信时拟合出的 **T > 1**（且接近锐化因子）；(b) 缩放后 **ECE 下降**；(c) **argmax 不变 → accuracy 不变**。

In [ ]:
def nll(probs, y):
    return -np.mean(np.log(probs[np.arange(len(y)), y] + 1e-12))

def fit_temperature(logits_val, y_val, grid=None):
    '''一维网格搜索 T 最小化验证集 NLL。'''
    if grid is None:
        grid = np.linspace(0.5, 5.0, 200)
    nlls = [nll(softmax(logits_val / T), y_val) for T in grid]
    return float(grid[int(np.argmin(nlls))])

# 用同分布的验证集拟合 T（真值 sharpen=2.5）
logits_val, y_val, _ = make_overconfident(n=6000, sharpen=2.5, seed=1)
T = fit_temperature(logits_val, y_val)
print(f'拟合温度 T = {T:.3f}  (真实锐化因子 = 2.5)')

# 在测试集上对比缩放前后
probs_before = softmax(logits)
probs_after  = softmax(logits / T)
conf_b, pred_b = probs_before.max(1), probs_before.argmax(1)
conf_a, pred_a = probs_after.max(1),  probs_after.argmax(1)
corr_b = (pred_b == y).astype(float); corr_a = (pred_a == y).astype(float)
ece_before = ece(conf_b, corr_b); ece_after = ece(conf_a, corr_a)
print(f'ECE: 缩放前 {ece_before:.4f}  ->  缩放后 {ece_after:.4f}')
print(f'accuracy: 缩放前 {corr_b.mean():.4f}  缩放后 {corr_a.mean():.4f}')
assert T > 1.0, '过度自信应拟合出 T>1'
assert ece_after < ece_before, '温度缩放应降低 ECE'
assert np.array_equal(pred_a, pred_b), 'T>0 时 argmax 不变 -> 预测/accuracy 完全不变'
print('✅ 温度缩放: T>1 把过度自信拉平、ECE 大降、且 accuracy 一动不动 —— 只调诚实度')

## 5 · Brier 分数与 Murphy 三分解

Brier = 概率预测的均方误差（恰当评分规则）。Murphy 把它**精确**分解为：
$$\mathrm{Brier}_{\text{分箱}} = \underbrace{\mathrm{rel}}_{\text{校准差}} - \underbrace{\mathrm{res}}_{\text{区分度}} + \underbrace{\mathrm{unc}}_{\text{固有难度}}$$

**关键**：这个等式对「把每个样本的预测替换成其所在箱的平均」后的**分箱 Brier** 精确成立。我们在「正确性 vs 置信度」的二分问题上验证它。

In [ ]:
def brier_binary(conf, correct):
    '''二分框架: 预测=置信度, 结果=是否正确。Brier = mean((conf - correct)^2)。'''
    return float(np.mean((conf - correct) ** 2))

def brier_decomposition(conf, correct, n_bins=15):
    '''Murphy 三分解。返回 (rel, res, unc, brier_binned)。
       brier_binned = 把每样本 conf 换成其箱内平均 后的 Brier，等式对它精确成立。'''
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    N = len(correct); acc_overall = correct.mean()
    rel = res = 0.0
    conf_binned = np.empty_like(conf)
    for b in range(n_bins):
        m = idx == b; n_b = m.sum()
        if n_b == 0:
            continue
        conf_b = conf[m].mean(); acc_b = correct[m].mean()
        rel += n_b / N * (conf_b - acc_b) ** 2
        res += n_b / N * (acc_b - acc_overall) ** 2
        conf_binned[m] = conf_b              # 用箱均值替换
    unc = acc_overall * (1 - acc_overall)
    brier_binned = float(np.mean((conf_binned - correct) ** 2))
    return rel, res, unc, brier_binned

rel, res, unc, brier_binned = brier_decomposition(conf, correct, n_bins=15)
raw_brier = brier_binary(conf, correct)
print(f'rel(校准差) = {rel:.5f}   res(区分度) = {res:.5f}   unc(固有难度) = {unc:.5f}')
print(f'rel - res + unc      = {rel - res + unc:.6f}')
print(f'分箱 Brier           = {brier_binned:.6f}')
print(f'原始(逐样本) Brier   = {raw_brier:.6f}  (与分箱差 = 箱内方差)')
assert abs((rel - res + unc) - brier_binned) < 1e-9, 'Murphy 分解对分箱 Brier 应精确成立'
print('✅ Brier = 校准差 - 区分度 + 固有难度，三项相加精确等于分箱 Brier')

## 6 · 选择性预测：risk-coverage 曲线

校准的回报：**没把握就弃答**。按置信度从高到低作答，覆盖率(作答比例) c 下的 risk(错误率) 怎么变？

好的模型：只答最有把握的少数时风险很低，随覆盖率升高风险逐步逼近整体错误率。

In [ ]:
def risk_coverage(conf, correct, coverages=None):
    '''按置信度降序，返回各覆盖率下的 (coverage, risk)。'''
    if coverages is None:
        coverages = np.linspace(0.1, 1.0, 10)
    order = np.argsort(-conf)            # 置信度从高到低
    corr_sorted = correct[order]
    N = len(correct); out = []
    for cov in coverages:
        k = max(1, int(round(cov * N)))
        risk = 1 - corr_sorted[:k].mean()  # 前 k 个(最自信)的错误率
        out.append((k / N, risk))
    return out

rc = risk_coverage(conf, correct)
print(f"{'覆盖率':>8} {'风险(错误率)':>12}")
for cov, risk in rc:
    print(f'{cov:>8.2f} {risk:>12.4f}')
risk_low_cov = rc[0][1]      # 只答最自信的 10%
risk_full    = rc[-1][1]     # 全部作答
print(f'\n只答最自信 10% 的风险 = {risk_low_cov:.4f}  vs  全部作答风险 = {risk_full:.4f}')
assert risk_low_cov <= risk_full, '最自信子集的错误率应 <= 整体错误率'
# 风险大体随覆盖率单调上升(允许小抖动)
risks = [r for _, r in rc]
assert risks[-1] >= risks[0], '覆盖率升高，累计风险总体上升'
print('✅ 弃答最没把握的样本能显著降低风险 —— 校准让「不确定性」变成可用信号')

---
## ✏️ 练习 1：任意箱数的 ECE

实现 `ece_nbins(conf, correct, n_bins)`：等宽分箱算 ECE（=各箱 |acc−conf| 按样本加权平均）。

用途：复现第 2 节，但你自己从头写一遍分箱逻辑（不调用第 2 节的函数）。

In [ ]:
def ece_nbins(conf, correct, n_bins=10):
    # TODO: 等宽分箱(np.linspace + np.digitize)，对每个非空箱累加 n_b/N*|acc_b-conf_b|
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng_ex = np.random.default_rng(7)
lg, yy, _ = make_overconfident(n=4000, sharpen=2.0, seed=7)
pb = softmax(lg); cf = pb.max(1); cr = (pb.argmax(1) == yy).astype(float)
e_mine = ece_nbins(cf, cr, n_bins=15)
e_ref  = ece(cf, cr, n_bins=15)            # 第 2 节的实现当参考
print(f'我的 ECE = {e_mine:.4f}   参考 ECE = {e_ref:.4f}')
assert abs(e_mine - e_ref) < 1e-9, '应与第 2 节实现逐位一致'
assert e_mine > 0.03, '过度自信 -> ECE>0'
print('✅ 练习 1 通过：ECE 分箱从零实现，与参考对拍一致')

## ✏️ 练习 2：可靠性图数据

实现 `reliability_curve(conf, correct, n_bins)`：返回 numpy 数组 `(bin_conf, bin_acc, bin_count)`（每个非空箱一行）。

这是画可靠性图所需的全部数据。

In [ ]:
def reliability_curve(conf, correct, n_bins=10):
    # TODO: 等宽分箱，每个非空箱返回 [平均置信度, 准确率, 样本数]
    #       返回三个等长 1D 数组 (bin_conf, bin_acc, bin_count)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
bc, ba, bn = reliability_curve(cf, cr, n_bins=10)
assert len(bc) == len(ba) == len(bn)
assert bn.sum() == len(cf), '各箱样本数之和应=总样本数'
assert np.all((bc >= 0) & (bc <= 1)) and np.all((ba >= 0) & (ba <= 1))
# 过度自信: 平均置信度(按样本加权) 应高于 平均准确率
w_conf = (bc * bn).sum() / bn.sum()
assert w_conf > (ba * bn).sum() / bn.sum(), '加权置信度应>加权准确率(过度自信)'
print(f'箱数={len(bc)}, 加权置信度={w_conf:.3f} > 加权准确率={(ba*bn).sum()/bn.sum():.3f}')
print('✅ 练习 2 通过：可靠性图数据正确（样本数守恒、过度自信可见）')

## ✏️ 练习 3：温度缩放

实现 `apply_temperature_and_ece(logits_test, y_test, logits_val, y_val, n_bins)`：
在验证集上拟合 T（最小化 NLL），在测试集上返回 `(T, ece_before, ece_after, acc_unchanged)`，其中 `acc_unchanged` 是 bool（缩放前后 argmax 是否完全相同）。可复用第 4 节的 `fit_temperature`/`softmax`/`ece`。

In [ ]:
def apply_temperature_and_ece(logits_test, y_test, logits_val, y_val, n_bins=15):
    # TODO: 1) T = fit_temperature(logits_val, y_val)
    #       2) 缩放前后各算 conf/pred/correct 与 ECE
    #       3) 返回 (T, ece_before, ece_after, 缩放前后 argmax 是否完全相同)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
lg_te, y_te, _ = make_overconfident(n=5000, sharpen=3.0, seed=11)
lg_va, y_va, _ = make_overconfident(n=5000, sharpen=3.0, seed=12)
T, eb, ea, unchanged = apply_temperature_and_ece(lg_te, y_te, lg_va, y_va)
print(f'T={T:.3f} (真值3.0)  ECE {eb:.4f} -> {ea:.4f}  argmax不变={unchanged}')
assert T > 1.5, '锐化因子3.0 应拟合出明显>1 的 T'
assert ea < eb, '温度缩放应降低 ECE'
assert unchanged, 'T>0 时 argmax 必须不变'
print('✅ 练习 3 通过：温度缩放降 ECE、不动 accuracy、T 接近真实锐化因子')

## ✏️ 练习 4：Brier 三分解

实现 `brier_three_parts(conf, correct, n_bins)`：返回 `(rel, res, unc, brier_binned)`，并满足 `rel − res + unc == brier_binned`（机器精度）。其中 brier_binned 是把每样本置信度替换成箱均值后的 Brier。

In [ ]:
def brier_three_parts(conf, correct, n_bins=15):
    # TODO: 分箱; rel=Σn_b/N*(conf_b-acc_b)^2; res=Σn_b/N*(acc_b-acc̄)^2; unc=acc̄(1-acc̄)
    #       brier_binned = mean((把conf换成箱均值 - correct)^2)
    #       返回 (rel, res, unc, brier_binned)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rel, res, unc, bb = brier_three_parts(cf, cr, n_bins=15)
print(f'rel={rel:.5f} res={res:.5f} unc={unc:.5f}')
print(f'rel-res+unc={rel-res+unc:.6f}  分箱Brier={bb:.6f}')
assert abs((rel - res + unc) - bb) < 1e-9, 'Murphy 分解必须精确成立'
assert rel >= 0 and res >= 0 and 0 <= unc <= 0.25, '各分量取值范围合理'
print('✅ 练习 4 通过：Brier 三分解精确(rel-res+unc=分箱Brier)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def ece_nbins(conf, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    N = len(conf); e = 0.0
    for b in range(n_bins):
        m = idx == b; n_b = m.sum()
        if n_b == 0:
            continue
        e += n_b / N * abs(correct[m].mean() - conf[m].mean())
    return e

In [ ]:
# 练习 2 参考答案
def reliability_curve(conf, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    bc, ba, bn = [], [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        bc.append(conf[m].mean()); ba.append(correct[m].mean()); bn.append(int(m.sum()))
    return np.array(bc), np.array(ba), np.array(bn)

In [ ]:
# 练习 3 参考答案
def apply_temperature_and_ece(logits_test, y_test, logits_val, y_val, n_bins=15):
    T = fit_temperature(logits_val, y_val)
    pb = softmax(logits_test); pa = softmax(logits_test / T)
    cb = (pb.argmax(1) == y_test).astype(float); ca = (pa.argmax(1) == y_test).astype(float)
    eb = ece(pb.max(1), cb, n_bins); ea = ece(pa.max(1), ca, n_bins)
    unchanged = bool(np.array_equal(pa.argmax(1), pb.argmax(1)))
    return T, eb, ea, unchanged

In [ ]:
# 练习 4 参考答案
def brier_three_parts(conf, correct, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    N = len(correct); acc_overall = correct.mean()
    rel = res = 0.0; conf_binned = np.empty_like(conf)
    for b in range(n_bins):
        m = idx == b; n_b = m.sum()
        if n_b == 0:
            continue
        conf_b = conf[m].mean(); acc_b = correct[m].mean()
        rel += n_b / N * (conf_b - acc_b) ** 2
        res += n_b / N * (acc_b - acc_overall) ** 2
        conf_binned[m] = conf_b
    unc = acc_overall * (1 - acc_overall)
    return rel, res, unc, float(np.mean((conf_binned - correct) ** 2))

---
## 🧪 真实数据胶囊：真实分类器的校准与温度缩放

用 UCI **wine-quality** 数据训一个**纯 numpy 的逻辑回归**（红酒好/坏二分类），得到真实预测概率，算它的 ECE，再用温度缩放修。

**联网取真实数据；失败回退到内置真实风格的 (logits, labels)。**

In [ ]:
def load_wine_logits(seed=0):
    '''取真实 wine-quality 数据训纯numpy逻辑回归 -> 返回 (logits Nx2, y, source)。
       失败回退到内置真实风格数值。'''
    def train_logreg(X, y, seed=0, iters=300, lr=0.1):
        r = np.random.default_rng(seed)
        Xb = np.hstack([X, np.ones((len(X), 1))])           # 加偏置
        w = r.normal(0, 0.01, Xb.shape[1])
        for _ in range(iters):
            z = Xb @ w; pr = 1 / (1 + np.exp(-z))
            w -= lr * Xb.T @ (pr - y) / len(y)
        z = Xb @ w
        return np.stack([-z / 2, z / 2], axis=1)            # 两类 logits(差=z)
    try:
        path = _get('https://archive.ics.uci.edu/ml/machine-learning-databases/'
                    'wine-quality/winequality-red.csv', 'winequality-red.csv')
        df = pd.read_csv(path, sep=';')
        X = df.drop(columns=['quality']).to_numpy(dtype=float)
        X = (X - X.mean(0)) / (X.std(0) + 1e-9)             # 标准化
        y = (df['quality'].to_numpy() >= 6).astype(float)  # 好酒(>=6) vs 不好
        logits = train_logreg(X, y, seed=seed)
        return logits, y.astype(int), 'online'
    except Exception as e:
        print('  (联网失败，回退内置:', type(e).__name__, ')')
        r = np.random.default_rng(seed)
        # 内置: 真实风格的中等区分度二分类 + 轻度过度自信
        n = 1500; true_logit = r.normal(0, 1.2, n)
        y = (r.random(n) < 1 / (1 + np.exp(-true_logit))).astype(int)
        z = true_logit * 1.6                                # 轻度过度自信
        return np.stack([-z / 2, z / 2], axis=1), y, 'builtin'

logits_real, y_real, src = load_wine_logits(seed=0)
print(f'数据来源 = {src}; 样本数 = {len(y_real)}')
pr = softmax(logits_real); cf_r = pr.max(1); cr_r = (pr.argmax(1) == y_real).astype(float)
print(f'真实分类器: accuracy = {cr_r.mean():.3f}, ECE = {ece(cf_r, cr_r):.4f}')
assert len(y_real) > 100 and pr.shape[1] == 2
print('✅ 在真实(或回退)数据上得到一个分类器的预测概率与校准')

**🧪 胶囊练习**：实现 `calibrate_real(logits, y, n_bins)`——把数据对半切成验证/测试，在验证半上拟合温度 T，返回 `(T, ece_test_before, ece_test_after)`，并应满足 `ece_after <= ece_before`（温度缩放不该让真实数据更差）。

In [ ]:
def calibrate_real(logits, y, n_bins=10):
    # TODO: 1) 对半切 val/test (前半 val, 后半 test)
    #       2) T = fit_temperature(logits_val, y_val)
    #       3) 在 test 上算缩放前/后 ECE，返回 (T, ece_before, ece_after)
    raise NotImplementedError

In [ ]:
# 自测
T, eb, ea = calibrate_real(logits_real, y_real, n_bins=10)
print(f'真实数据: T={T:.3f}  测试集 ECE {eb:.4f} -> {ea:.4f}')
assert T > 0
assert ea <= eb + 0.02, '温度缩放在真实数据上不应明显恶化校准'
print('✅ 胶囊练习通过：在真实数据上拟合温度并改善(或不恶化)校准')

In [ ]:
# 📖 胶囊参考答案
def calibrate_real(logits, y, n_bins=10):
    h = len(y) // 2
    lv, yv = logits[:h], y[:h]; lt, yt = logits[h:], y[h:]
    T = fit_temperature(lv, yv)
    pb = softmax(lt); pa = softmax(lt / T)
    cb = (pb.argmax(1) == yt).astype(float); ca = (pa.argmax(1) == yt).astype(float)
    return T, ece(pb.max(1), cb, n_bins), ece(pa.max(1), ca, n_bins)

### 小结
- **校准 vs 准确率正交**：准确率问答对多少，校准问「它说的『我确定』有多可信」。
- **可靠性图**：分箱画 conf vs acc，对角线下方=过度自信；**ECE**=平均垂直差距、**MCE**=最坏箱。
- 现代网络系统性**过度自信**（交叉熵+过参数化把 logits 推得过尖），主要是**尺度问题、不动 argmax**。
- **温度缩放**：softmax(z/T)，验证集上拟合 T，**降 ECE、零成本、不伤 accuracy**——首选事后校准。
- **Brier = 校准差(rel) − 区分度(res) + 固有难度(unc)**，三项相加精确等于分箱 Brier。
- 校准的回报是**选择性预测**：没把握就弃答，risk-coverage 量化「不确定性是否可用」。

下一站：**模块 07 · 在线 A/B 评测** —— 把评测从离线测试集搬到线上：假设检验、功效、序贯检验、CUPED。